<h1>🎛️ Biofilter — Report: <code>pair_genes</code></h1>

Which of these genes are related, and by what — and optionally, what that
implies about a list of your own.

Two stages: **connect** genes through a shared pathway, disease or
protein, then **expand** — only if you ask — by a gene → item mapping you
supply.

Section 4 is the one to read. It is why this report exists rather than
being a mode of `pair_variants`.

### 5. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

BUNDLE = None
REPORT = "pair_genes"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["CHEK2", "SMARCB1", "NF2"]
print(bf.core.db_uri)

### 6. Gene pairs, on their own

No mapping: the answer is which of your genes are related, and by what.
That question stands by itself, which is the argument for this being a
report rather than a parameter of another one.

In [ ]:
pairs = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"])
df = pairs.to_pandas()

print(f"{len(df)} gene pairs, tables: {list(pairs.tables)}")
df[["gene_1_symbol", "gene_2_symbol", "group_support_count",
    "group_support_source_count", "group_support_sources"]]

`group_support_sources` names the curation from the bundle rather than
guessing it from an accession prefix. Two curations agreeing is a
different claim from one curation saying it twice, which is what
`min_group_sources` filters on — and what `min_group_support` does not.

### 7. `max_group_size` decides the size *and* the meaning

A pathway naming 2,615 genes links its members while saying almost
nothing about any of them. When a result comes back empty or thin, this
is usually why — so the provenance says what the cut removed.

In [ ]:
for size in (200, 300, 1000, 0):
    out = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"],
                        max_group_size=size)
    label = "no limit" if size == 0 else str(size)
    print(f"  max_group_size={label:<9} {out.num_rows:>3} pairs")

pairs.provenance["group_filter"]

### 8. Why this is not a mode of `pair_variants`

`pair_variants` derives "this variant belongs to this gene" from
coordinates. That is right for a coding variant and wrong for a
regulatory one: a variant sits in one gene and acts on another.

Ask the bundle how often those differ.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    disagreement = bundle.con.execute("""
        WITH linked AS (
            SELECT DISTINCT
                g.chromosome, g.position, g.reference_allele, g.alternate_allele,
                g.gene_id AS regulated, e.gene AS sits_in
            FROM variant_gtex g
            JOIN variant_molecular_effects e
              ON e.chromosome = g.chromosome AND e.position = g.position
             AND e.reference_allele = g.reference_allele
             AND e.alternate_allele = g.alternate_allele
            WHERE e.gene IS NOT NULL AND g.gene_id IS NOT NULL
        )
        SELECT count(*) AS pairs,
               count(*) FILTER (WHERE regulated <> sits_in) AS different_gene
        FROM linked
    """).to_arrow_table().to_pandas()

share = disagreement.different_gene[0] / max(disagreement.pairs[0], 1)
print(f"{disagreement.pairs[0]:,} variant x gene links carrying both kinds of evidence")
print(f"{share:.1%} name a gene other than the one the variant sits in")

So when your evidence for the attachment comes from outside Biofilter —
a colocalization, a fine-mapping, a curated list — `pair_variants` cannot
use it: its stage 3 re-derives membership from coordinates and drops
anything that disagrees, silently.

`pair_genes` never derives it. It takes the link you supply.

### 9. How you name a gene

Three mechanisms, and you say which — for `input_data` and the mapping
alike, since it is one decision about one thing.

| `gene_identifier` | Looks in |
| --- | --- |
| `alias` (default) | every alias: symbols, synonyms, HGNC, Ensembl, Entrez |
| a code system — `HGNC`, `ENTREZ`, `ENSEMBL`, … | only that system |
| `entity_id` (or `biofilter_id`) | the bundle's key, skipping aliases |

This is a parameter rather than something the report works out, because
nothing can tell them apart by looking.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    collisions = bundle.con.execute("""
        SELECT count(*) AS numeric_aliases,
               count(*) FILTER (WHERE gm.entity_id IS NOT NULL) AS also_an_entity_id
        FROM entity_aliases a
        LEFT JOIN gene_masters gm
               ON gm.entity_id = TRY_CAST(a.alias_value AS BIGINT)
              AND gm.entity_id <> a.entity_id
        WHERE TRY_CAST(a.alias_value AS BIGINT) IS NOT NULL
    """).to_arrow_table().to_pandas()

print(f"{collisions.numeric_aliases[0]:,} aliases are bare numbers (Entrez ids)")
print(f"{collisions.also_an_entity_id[0]:,} of them are the entity id of a *different* gene")
print("\nEntrez 2 is A2M. Entity 2 is A1BG-AS1. A report that guessed would")
print("return the wrong gene and say nothing.")

In [ ]:
# The same three genes, named three ways.
with Bundle.open(bf.core.db_uri.removeprefix("parquet://")) as bundle:
    ids = bundle.con.execute(f"""
        SELECT gm.symbol, gm.entity_id,
               max(CASE WHEN a.xref_source='ENSEMBL' THEN a.alias_value END) AS ensembl
        FROM gene_masters gm
        JOIN entity_aliases a ON a.entity_id = gm.entity_id
        WHERE gm.symbol IN ('CHEK2', 'SMARCB1', 'NF2')
        GROUP BY 1, 2
    """).to_arrow_table().to_pandas()

for label, values, how in [
    ("symbols",     ids.symbol.tolist(),               None),
    ("Ensembl ids", ids.ensembl.tolist(),              "ensembl"),
    ("entity ids",  ids.entity_id.astype(str).tolist(), "entity_id"),
]:
    kw = {"gene_identifier": how} if how else {}
    out = bf.report.run(REPORT, input_data=values, group_types=["Proteins"], **kw)
    print(f"  {label:<12} -> {out.num_rows} gene pairs")

Naming the code system is also a **narrower** search, not just a
disambiguation: under `gene_identifier=entrez`, `2` can only be A2M
because no other column is consulted.

`entity_id` is the bundle's own key — exact, and scoped to the build that
issued it (ADR-003 §2.5). A list of entity ids belongs with the
`bundle_id` it came from.

### 10. The mapping: two columns, gene then item

Many-to-many in both directions. The worked example from ADR-005: three
items on one gene, two on the other, one pair between them.

In [ ]:
MAPPING = {
    "CHEK2":   ["111", "222", "333"],
    "SMARCB1": ["444", "555"],
}

expanded = bf.report.run(REPORT, input_data=["CHEK2", "SMARCB1"],
                         group_types=["Proteins"], mapping=MAPPING)

print(f"tables: {list(expanded.tables)}")
print(f"  result     {expanded.num_rows} item pairs   <- what write() exports")
print(f"  gene_pairs {expanded.extra_tables['gene_pairs'].num_rows} gene pair")
expanded.to_pandas()[["item_1", "item_2", "gene_1_symbol", "gene_2_symbol"]]

In [ ]:
# The same thing from a file, which is what a real mapping arrives as.
mapping_file = OUTPUT_DIR / "demo_mapping.tsv"
mapping_file.write_text("\n".join(
    f"{gene}\t{item}" for gene, items in MAPPING.items() for item in items) + "\n")

from_file = bf.report.run(REPORT, input_data=["CHEK2", "SMARCB1"],
                          group_types=["Proteins"], mapping_file=str(mapping_file))
print("same answer:", from_file.num_rows == expanded.num_rows)

### 11. The item is never read

Which is what makes gene→position, gene→rsID, gene→probe and
gene→exposure one feature instead of four.

**Biofilter is build 38 and managing build is yours** — but nothing here
interprets a coordinate, so build-37 positions pass through correctly and
Biofilter never has to know what one is.

In [ ]:
anything = bf.report.run(
    REPORT, input_data=["CHEK2", "SMARCB1"], group_types=["Proteins"],
    mapping={"CHEK2": ["17:7579472", "exposure:smoking"],
             "SMARCB1": ["probe_0042"]},
).to_pandas()

anything[["item_1", "item_2"]]

The price of that is real and worth stating: the report cannot filter by
allele frequency, resolve an rsID, or validate an item — and **two
spellings of one thing are two things**. `22:100:A:G` and
`chr22:100:A:G` are different items, and that bounds what the
deduplication below can promise.

### 12. Three rules the simple example does not show

The cross product is not the work. These are, and they are identical for
every caller — which is the argument for the platform owning them rather
than each analysis re-deriving them.

In [ ]:
# An item on both genes would otherwise pair with itself.
self_pair = bf.report.run(
    REPORT, input_data=["CHEK2", "SMARCB1"], group_types=["Proteins"],
    mapping={"CHEK2": ["X", "111"], "SMARCB1": ["X", "444"]},
).to_pandas()

print("pairs:", sorted(zip(self_pair.item_1, self_pair.item_2)))
print("X paired with itself:", bool((self_pair.item_1 == self_pair.item_2).any()))

In [ ]:
# Deduplication is global, not per gene pair: the same item pair arrives
# through every gene pair linking it. On one real run that was 4.3% of
# the answer — 72,554 against the 75,794 a naive sum(n1 x n2) reports.
three = bf.report.run(
    REPORT, input_data=GENES, group_types=["Proteins"],
    mapping={"CHEK2": ["A"], "SMARCB1": ["B"], "NF2": ["A"]},
)
print(f"{three.extra_tables['gene_pairs'].num_rows} gene pairs "
      f"-> {three.num_rows} item pair(s)")
three.to_pandas()[["item_1", "item_2"]]

### 13. Both genes must carry an item

Not "both were named". A gene can be in your input and carry nothing, and
then there is nothing on its side to pair.

For the same reason `membership="either"` is refused with a mapping: the
partner gene came from the bundle, not from your list.

In [ ]:
partial = bf.report.run(REPORT, input_data=GENES, group_types=["Proteins"],
                        mapping={"CHEK2": ["A"], "SMARCB1": ["B"]})
print(f"NF2 is in the input and carries nothing -> {partial.num_rows} pair(s)")

try:
    bf.report.run(REPORT, input_data=["CHEK2"], membership="either",
                  mapping={"CHEK2": ["A"]})
except ValueError as exc:
    print("\nrefused:", exc)

### 14. Export, and keeping both tables

`write()` exports the primary table, which is the item pairs when you
asked for them. `save()` keeps everything.

In [ ]:
for path in expanded.write(OUTPUT_DIR / "pair_genes.csv"):
    print(path)

saved = expanded.save(OUTPUT_DIR / "runs" / "pair_genes", overwrite=True)
print("\nsaved:", saved)

from biofilter.modules.report.result import ReportResult
back = ReportResult.load(saved)
print("tables back:", list(back.tables))

### 15. The same thing on the command line

```bash
biofilter report run --report-name pair_genes \\
    --input CHEK2 --input SMARCB1 --input NF2 \\
    --param group_types=Proteins \\
    --param max_group_size=300 \\
    --param mapping_file=./variant_to_gene.tsv \\
    --output item_pairs.csv
```